In [74]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd
from src.classic_ml.models.seBert_model import seBERT
import gensim
import torch
from transformers import BertModel, BertTokenizer
from src.classic_ml.models.seBert import seBERT_MultiTask
from transformers import BertTokenizer, BertModel
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

True
NVIDIA GeForce RTX 5090
12.8


In [33]:
df = pd.read_csv('../data/raw/github-labels-annotated.csv')

In [34]:
se_bert_model = seBERT(batch_size=16)
import torch
torch.cuda.is_available()

True

In [35]:
se_bert_model.model

seBERT_MultiTask(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((

In [36]:
y = df[["worth_automating", "enough_information"]]


In [37]:

# preprocess it
def preprocess(row):
    ret = str(row['issue_title']) + " " + str(row['issue_body'])
    ret = gensim.parsing.preprocessing.strip_multiple_whitespaces(ret)
    ret = ret.replace('\r\n', ' ')
    ret = ret.replace('\n', ' ')
    return ret

df['text_no_newlines'] = df.apply(preprocess, axis=1)
X = df['text_no_newlines'].tolist()



se_bert_model.fit(X, y)


c:\Users\user\Desktop\Practicas_Manuel\T3.8.4\T3.8.4\.venv\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Auroc Enough Info,Auprc Enough Info,F1 Enough Info,Macro F1 Enough Info,Auroc Worth Auto,Auprc Worth Auto,F1 Worth Auto,Macro F1 Worth Auto
1,0.695600,0.581419,0.951735,0.966650,0.907166,0.879749,0.944305,0.925202,0.842645,0.859880
2,0.383700,0.586169,0.953052,0.966976,0.909440,0.870071,0.953157,0.938756,0.862967,0.877321
3,0.163600,0.751050,0.950050,0.959460,0.910707,0.873613,0.952075,0.931989,0.859210,0.872183
4,0.061900,0.888520,0.940868,0.943206,0.910638,0.873694,0.951068,0.932059,0.862207,0.877572


=============TYPE =======================================
=============TYPE =======================================
=============TYPE =======================================
=============TYPE =======================================
TrainOutput(global_step=500, training_loss=0.3262164669036865, metrics={'train_runtime': 138.1878, 'train_samples_per_second': 231.569, 'train_steps_per_second': 3.618, 'total_flos': 0.0, 'train_loss': 0.3262164669036865, 'epoch': 4.0})


,checkpoints_dir,'../checkpoints/'
,batch_size,16


Evaluation

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_tokinizers = BertTokenizer.from_pretrained("../checkpoints/final_model")

base_model = BertModel.from_pretrained("../src/classic_ml/models/seBERT")
model = seBERT_MultiTask(base_model)

state_dict = torch.load("../checkpoints/final_model/model.pt", map_location=device)
model.load_state_dict(state_dict)

model.to(device)
model.eval()

seBERT_MultiTask(
  (encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((

In [43]:
test_tokinizers

BertTokenizer(name_or_path='../checkpoints/final_model', vocab_size=30522, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [60]:
df_test = pd.read_csv("../data/raw/github-labels-annotated.csv")

df_test["text_no_newlines"] = df_test.apply(preprocess, axis=1)
X_TEST = df["text_no_newlines"].values
y_TEST = df_test[["worth_automating", "enough_information"]]

data_len = len(df_test)
y_probs_worth_automating, y_probs_enough = [], []
model.eval()
with torch.no_grad():
    for _, X_row in enumerate(X_TEST):
        inputs = test_tokinizers(X_row, padding=True,truncation=True, max_length=128, return_tensors="pt").to(device)
        outputs = model(**inputs)       
        logits_worth_automating = torch.sigmoid(outputs["logits_worth_automating"])
        logits_enough_information = torch.sigmoid(outputs["logits_enough_information"])
        y_probs_worth_automating.append(logits_worth_automating.item())
        y_probs_enough.append(logits_enough_information.item())

In [66]:
y_pred_worth_automating, y_pred_enough = [], []
for y_p_w_a, y_p_e in zip(y_probs_worth_automating, y_probs_enough):
    y_pred_worth_automating.append(int(y_p_w_a >= 0.5))
    y_pred_enough.append(int(y_p_e >= 0.5))


In [ ]:
from sklearn.metrics import matthews_corrcoef, f1_score, precision_score, recall_score


y_test_worth = y_TEST["worth_automating"].values
y_test_enough = y_TEST["enough_information"].values

results = [{
    'model': 'seBERT - Worth Automating',
    'mcc': matthews_corrcoef(y_true=y_test_worth, y_pred=y_pred_worth_automating),
    'micro_f1': f1_score(y_true=y_test_worth, y_pred=y_pred_worth_automating, average='micro'),
    'micro_precision': precision_score(y_true=y_test_worth, y_pred=y_pred_worth_automating, average='micro'),
    'micro_recall': recall_score(y_true=y_test_worth, y_pred=y_pred_worth_automating, average='micro'),
    'macro_f1': f1_score(y_true=y_test_worth, y_pred=y_pred_worth_automating, average='macro'),
    'macro_precision': precision_score(y_true=y_test_worth, y_pred=y_pred_worth_automating, average='macro'),
    'macro_recall': recall_score(y_true=y_test_worth, y_pred=y_pred_worth_automating, average='macro'),
},
{
    'model': 'seBERT - Enough Information',
    'mcc': matthews_corrcoef(y_true=y_test_enough, y_pred=y_pred_enough),
    'micro_f1': f1_score(y_true=y_test_enough, y_pred=y_pred_enough, average='micro'),
    'micro_precision': precision_score(y_true=y_test_enough, y_pred=y_pred_enough, average='micro'),
    'micro_recall': recall_score(y_true=y_test_enough, y_pred=y_pred_enough, average='micro'),
    'macro_f1': f1_score(y_true=y_test_enough, y_pred=y_pred_enough, average='macro'),
    'macro_precision': precision_score(y_true=y_test_enough, y_pred=y_pred_enough, average='macro'),
    'macro_recall': recall_score(y_true=y_test_enough, y_pred=y_pred_enough, average='macro'),
}]

results_df = pd.DataFrame(results)
results_df

,model,mcc,micro_f1,micro_precision,micro_recall,macro_f1,macro_precision,macro_recall
0,seBERT - Worth Automating,0.945535,0.9732,0.9732,0.9732,0.972734,0.972038,0.973498
1,seBERT - Enough Information,0.944139,0.9737,0.9737,0.9737,0.971891,0.975333,0.968829


In [77]:
test_issues = [
    {
        "title": "App crashes when uploading files larger than 10MB",
        "body": "Steps to reproduce:\n1. Go to upload section\n2. Select file > 10MB\n3. App crashes with OutOfMemoryError"
    },
    {
        "title": "Rethink the whole UI design",
        "body": "I think we should discuss changing the entire design system"
    },
    {
        "title": "Fix login button",
        "body": ""
    }
]

for issue in test_issues:
    pred_worth, pred_enough = se_bert_model.predict([issue["title"] + " " + issue["body"]])
    print(f"\nTitle: {issue['title']}")
    print(f"  worth_automating:  {pred_worth[0]}")
    print(f"  enough_information: {pred_enough[0]}")


Title: App crashes when uploading files larger than 10MB
  worth_automating:  1
  enough_information: 1

Title: Rethink the whole UI design
  worth_automating:  0
  enough_information: 0

Title: Fix login button
  worth_automating:  0
  enough_information: 0
